In [227]:
# Install Dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [228]:
from dotenv import load_dotenv

load_dotenv()

True

In [229]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"


In [230]:
def addUserMsg(messages, text):
    msg = {"role":"user", "content":text}
    messages.append(msg)

def addAssistantMsg(messages, text):
    msg = {"role":"assistant", "content":text}
    messages.append(msg)

def sendMsg(messages, text, systemPrompt=None, startFrom=None, stop_sequence=None):
    addUserMsg(messages,text)
    if startFrom:
        addAssistantMsg(messages, startFrom)
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
    }
    if stop_sequence:
        params["stop_sequences"] = [stop_sequence]
    if systemPrompt:
        params["system"] = systemPrompt
    response = client.messages.create(**params)
    respText = response.content[0].text
    addAssistantMsg(messages,respText)
    return respText


In [231]:
messages = []

sendMsg(messages, "What is quantum computing. Write in 1 sentence")


'Quantum computing is a revolutionary technology that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that allow certain complex problems to be solved exponentially faster than classical computers.'

In [232]:
sendMsg(messages, "Give another sentence")

'Quantum computers leverage quantum bits (qubits) that can exist in multiple states simultaneously, enabling them to perform many calculations in parallel and tackle problems in cryptography, drug discovery, and optimization that are practically impossible for traditional computers.'

In [233]:
chat = []
while True:
    try:
        inp = input("Question?")
        print(">"+inp)
        r = sendMsg(chat, inp)
        print("----")
        print(r)
        print("----")
    except KeyboardInterrupt:
        break


In [234]:
sendMsg([], "What is value of 4x+7=9 ?")

'I need to solve for x in the equation 4x + 7 = 9.\n\n**Step 1:** Subtract 7 from both sides\n4x + 7 - 7 = 9 - 7\n4x = 2\n\n**Step 2:** Divide both sides by 4\n4x/4 = 2/4\nx = 1/2\n\n**Answer:** x = 1/2 (or 0.5)\n\nTo verify: 4(1/2) + 7 = 2 + 7 = 9 ✓'

In [235]:
systemPrompt = """
You are a teacher.
Initially give hints rather than complete solutions.
Patiently walk students through problems step by step.
Show solutions for similar problems as examples.
"""
sendMsg([], "What is value of 4x+7=9 ?", systemPrompt)

'I\'ll help you solve for x in the equation **4x + 7 = 9**.\n\nLet me give you a hint to get started:\n\n**Hint:** To isolate x, you first need to get the term with x by itself on one side. What do you think should be your first step to remove the 7 from the left side?\n\nThink about what operation would "undo" adding 7.\n\n---\n\nGive it a try, and let me know what you think the first step should be! 😊'

In [236]:
sendMsg([], "Generate a 1 sentence Movie Idea")

'A washed-up astronaut discovers his teenage daughter has been secretly building a rocket in their barn to finish the Mars mission he abandoned years ago.'

In [237]:
def sendStream(messages, text, callback):
    addUserMsg(messages, text)
    with client.messages.stream(
            model=model,
            max_tokens=1000,
            messages=messages
    ) as stream:
        for text in stream.text_stream:
            callback(text)
        return stream.get_final_message().content[0].text


In [238]:
sendStream([], "Generate 1 sentence about planets", lambda t: print(t)) #, end=""))

The
 eight
 planets in our solar system orbit
 the Sun
 at
 varying
 distances,
 from
 scor
ching Mercury to frig
id Neptune
.


'The eight planets in our solar system orbit the Sun at varying distances, from scorching Mercury to frigid Neptune.'

In [239]:
sendMsg([], "Rule to monitor EC2 instances as json code block")

'```json\n{\n  "Version": "2012-10-17",\n  "Statement": [\n    {\n      "Sid": "EC2MonitoringPermissions",\n      "Effect": "Allow",\n      "Action": [\n        "ec2:DescribeInstances",\n        "ec2:DescribeInstanceStatus",\n        "ec2:DescribeVolumes",\n        "ec2:DescribeVolumeStatus",\n        "ec2:DescribeSnapshots",\n        "ec2:DescribeTags",\n        "ec2:DescribeRegions",\n        "ec2:DescribeAvailabilityZones",\n        "cloudwatch:GetMetricStatistics",\n        "cloudwatch:ListMetrics",\n        "cloudwatch:DescribeAlarms",\n        "cloudwatch:DescribeAlarmsForMetric"\n      ],\n      "Resource": "*"\n    }\n  ]\n}\n```\n\nThis IAM policy provides read-only permissions to monitor EC2 instances and their associated CloudWatch metrics. It allows:\n\n- Viewing instance details and status\n- Checking volume and snapshot information\n- Accessing CloudWatch metrics and alarms\n- Describing regions and availability zones'

In [240]:
js = sendMsg([], "Rule to monitor EC2 instances as json code block", startFrom="```json", stop_sequence="```")
import json

json.loads(js.strip())

{'Version': '2012-10-17',
 'Statement': [{'Effect': 'Allow',
   'Action': ['ec2:DescribeInstances',
    'ec2:DescribeInstanceStatus',
    'ec2:DescribeVolumes',
    'ec2:DescribeVolumeStatus',
    'ec2:DescribeSnapshots',
    'ec2:DescribeRegions',
    'ec2:DescribeAvailabilityZones',
    'ec2:DescribeTags',
    'cloudwatch:GetMetricStatistics',
    'cloudwatch:ListMetrics',
    'cloudwatch:PutMetricData',
    'cloudwatch:DescribeAlarms',
    'logs:CreateLogGroup',
    'logs:CreateLogStream',
    'logs:PutLogEvents',
    'logs:DescribeLogStreams'],
   'Resource': '*'}]}

In [244]:
sendMsg([], "generate 3 basic aws cli commands. Each should be very short")

'Here are 3 basic AWS CLI commands:\n\n```bash\n# List all S3 buckets\naws s3 ls\n\n# List EC2 instances\naws ec2 describe-instances\n\n# Get current IAM user\naws sts get-caller-identity\n```'

In [246]:
sendMsg([], "generate 3 basic aws cli commands. Each should be very short", startFrom="Here are the 3 commands in a single block without comments: \n```bash", stop_sequence="```")

'\naws s3 ls\naws ec2 describe-instances\naws iam list-users\n'